In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, load_npz

os.makedirs('../data/features', exist_ok=True)

In [ ]:
# Load row index
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

# Load feature blocks
X_tags    = load_npz('../data/features/album_tags_matrix.npz')
X_labels  = load_npz('../data/features/album_labels_matrix.npz')
X_types   = load_npz('../data/features/album_types_matrix.npz')
X_ratings = load_npz('../data/features/album_ratings_matrix.npz')

# Expand to full album universe
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

if len(album_id_order) < len(full_album_ids):
    print(f"Expanding matrices from {len(album_id_order):,} → {len(full_album_ids):,} albums...")
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    X_tags    = _expand(X_tags,    current_pos, n_full)
    X_labels  = _expand(X_labels,  current_pos, n_full)
    X_types   = _expand(X_types,   current_pos, n_full)
    X_ratings = _expand(X_ratings, current_pos, n_full)
    album_id_order = full_album_ids.tolist()

# Assemble final matrix
X_final_album_knn = hstack([X_tags, X_labels, X_types, X_ratings]).tocsr()

print(f"X_final_album_knn: {X_final_album_knn.shape[0]:,} albums x {X_final_album_knn.shape[1]:,} features  (nnz={X_final_album_knn.nnz:,})")
print(f"album_id_order length: {len(album_id_order):,}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# --- per-column stats across the full matrix ---
col_nnz   = np.diff(X_final_album_knn.tocsc().indptr)   # non-zeros per column
col_sums  = np.asarray(X_final_album_knn.sum(axis=0)).ravel()
n_albums  = X_final_album_knn.shape[0]
col_density = col_nnz / n_albums  # fraction of albums with a non-zero value

print(f"Total columns : {len(col_nnz):,}")
print(f"Zero columns  : {(col_nnz == 0).sum():,}")
print(f"Singleton cols: {(col_nnz == 1).sum():,}  (exactly 1 album)")
print(f"Columns with ≤5 albums  : {(col_nnz <= 5).sum():,}")
print(f"Columns with ≤10 albums : {(col_nnz <= 10).sum():,}")
print(f"\ncol_nnz percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{p:>2}: {np.percentile(col_nnz, p):.0f}")

In [ ]:
# --- distribution of column nnz (log scale) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(col_nnz, bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Albums per column (nnz)')
axes[0].set_ylabel('Number of columns')
axes[0].set_title('Column nnz distribution (linear)')

axes[1].hist(col_nnz[col_nnz > 0], bins=100, color='steelblue', edgecolor='none', log=True)
axes[1].set_xscale('log')
axes[1].set_xlabel('Albums per column (nnz, log scale)')
axes[1].set_ylabel('Number of columns (log scale)')
axes[1].set_title('Column nnz distribution (log-log)')

plt.tight_layout()
plt.show()

In [ ]:
# --- per-block breakdown ---
block_sizes = {
    'X_tags':    X_tags.shape[1],
    'X_labels':  X_labels.shape[1],
    'X_types':   X_types.shape[1],
    'X_ratings': X_ratings.shape[1],
}

block_cols = {}
start = 0
for name, size in block_sizes.items():
    block_cols[name] = col_nnz[start:start + size]
    start += size

print(f"{'Block':<12} {'cols':>6}  {'zero':>6}  {'≤5 nnz':>7}  {'median nnz':>10}  {'max nnz':>10}")
print("-" * 60)
for name, nnz in block_cols.items():
    print(f"{name:<12} {len(nnz):>6,}  {(nnz==0).sum():>6,}  {(nnz<=5).sum():>7,}  "
          f"{np.median(nnz):>10.0f}  {nnz.max():>10,}")

In [ ]:
# --- column density CDF — shows what threshold captures what share of columns ---
thresholds = [1, 2, 5, 10, 25, 50, 100, 250, 500]
print("Min-nnz threshold  |  columns kept  |  % of total")
print("-" * 50)
for t in thresholds:
    kept = (col_nnz >= t).sum()
    print(f"  ≥ {t:<4}            |  {kept:>6,}         |  {kept/len(col_nnz)*100:.1f}%")

fig, ax = plt.subplots(figsize=(9, 4))
sorted_nnz = np.sort(col_nnz)
ax.plot(sorted_nnz, np.linspace(0, 1, len(sorted_nnz)), color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('Minimum nnz threshold (log scale)')
ax.set_ylabel('Fraction of columns retained')
ax.set_title('CDF: columns retained vs. min-nnz cut')
ax.axvline(10,  color='orange', linestyle='--', label='nnz=10')
ax.axvline(50,  color='red',    linestyle='--', label='nnz=50')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# For each album, find the highest col_nnz among its non-zero columns.
# If that max < threshold t, the album becomes a zero row after pruning.
# Uses np.maximum.reduceat to avoid building a second sparse matrix.
col_nnz_vals = col_nnz[X_final_album_knn.indices]  # col_nnz for every stored entry

row_lengths = np.diff(X_final_album_knn.indptr)
nonempty_rows = np.where(row_lengths > 0)[0]
row_starts = X_final_album_knn.indptr[nonempty_rows]

max_col_nnz_per_album = np.zeros(n_albums, dtype=col_nnz.dtype)
max_col_nnz_per_album[nonempty_rows] = np.maximum.reduceat(col_nnz_vals, row_starts)

already_empty = (max_col_nnz_per_album == 0).sum()
print(f"Albums already with no features: {already_empty:,}  ({already_empty/n_albums*100:.2f}%)\n")

thresholds = [1, 2, 5, 10, 25, 50, 100, 250, 500]
print(f"{'threshold':>10} | {'cols kept':>10} | {'cols %':>7} | {'albums zeroed':>14} | {'albums %':>9} | {'newly zeroed':>13}")
print("-" * 75)
for t in thresholds:
    kept_cols = (col_nnz >= t).sum()
    zeroed    = (max_col_nnz_per_album < t).sum()
    newly     = zeroed - already_empty
    print(f"{t:>10} | {kept_cols:>10,} | {kept_cols/len(col_nnz)*100:>6.1f}% | "
          f"{zeroed:>14,} | {zeroed/n_albums*100:>8.3f}% | {newly:>13,}")

In [ ]:
# Boolean mask: which albums have at least one feature
has_features = row_lengths > 0

print(f"Albums with features    : {has_features.sum():>10,}  ({has_features.mean()*100:.1f}%)")
print(f"Albums without features : {(~has_features).sum():>10,}  ({(~has_features).mean()*100:.1f}%)")
print(f"Total                   : {len(has_features):>10,}")
print()
print("At query time:")
print("  - If the query album has no features → return 'no recommendations available'")
print("  - Only albums with features can be used as query seeds")

In [ ]:
# Find the highest threshold that zeroes no additional albums.
# = the minimum "best column nnz" across all albums that have features.
safe_threshold = int(max_col_nnz_per_album[has_features].min())

keep_cols = col_nnz >= safe_threshold
X_knn = X_final_album_knn[:, keep_cols]

print(f"Safe threshold : {safe_threshold} (no album with features loses all its features)")
print(f"Columns before : {X_final_album_knn.shape[1]:,}")
print(f"Columns after  : {X_knn.shape[1]:,}  ({keep_cols.sum()/len(col_nnz)*100:.1f}% retained)")
print(f"nnz before     : {X_final_album_knn.nnz:,}")
print(f"nnz after      : {X_knn.nnz:,}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# Subset to albums with features only — zero rows add no signal and slow down brute-force search
X_knn_annotated    = X_knn[has_features].copy()
album_ids_annotated = np.array(album_id_order)[has_features]

# Remove any NaN values stored explicitly in the sparse data array
nan_count = np.isnan(X_knn_annotated.data).sum()
if nan_count:
    print(f"Removing {nan_count:,} NaN entries from sparse data...")
    np.nan_to_num(X_knn_annotated.data, nan=0.0, copy=False)
    X_knn_annotated.eliminate_zeros()

# L2-normalise so cosine similarity reduces to a dot product (faster at query time)
X_knn_norm = normalize(X_knn_annotated, norm='l2')

print(f"Fitting on {X_knn_norm.shape[0]:,} albums x {X_knn_norm.shape[1]:,} features")

In [ ]:
model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model.fit(X_knn_norm)
print("Model fitted.")

In [ ]:
# Sanity check — query the first annotated album and print its 10 nearest neighbours
distances, indices = model.kneighbors(X_knn_norm[0], n_neighbors=11)

print(f"Query album id : {album_ids_annotated[0]}")
print(f"\n{'rank':<6} {'album_id':<40} {'cosine distance':>15}")
print("-" * 62)
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    label = "(query)" if rank == 0 else ""
    print(f"{rank:<6} {str(album_ids_annotated[idx]):<40} {dist:>15.4f}  {label}")

In [ ]:
import joblib
from scipy.sparse import save_npz

os.makedirs('../data/model', exist_ok=True)

joblib.dump(model,               '../data/model/knn_model.joblib')
save_npz('../data/model/X_knn_norm.npz', X_knn_norm)
np.save('../data/model/album_ids_annotated.npy', album_ids_annotated)
np.save('../data/model/has_features.npy',        has_features)

print("Saved:"             )
print("  ../data/model/knn_model.joblib")
print("  ../data/model/X_knn_norm.npz")
print("  ../data/model/album_ids_annotated.npy")
print("  ../data/model/has_features.npy")